# 配套实践 09-02：Transformer 怎样识别事件顺序

本练习构造一项最小机器人历史任务：序列中各出现一次“接触物体”和“抬起物体”，模型需要判断接触是否发生在抬起之前。两类样本包含完全相同的事件，只有先后顺序不同，因此可以直接检验位置编码的作用。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/09-attention-and-transformer/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import warnings  # 控制 PyTorch 在教学环境中的非关键提示
import numpy as np  # 整理预测结果与混淆矩阵
import torch  # 构造张量并训练小型 Transformer
from torch import nn  # 使用线性层、Transformer 编码器和损失函数
import matplotlib.pyplot as plt  # 绘制事件序列、训练曲线和混淆矩阵
warnings.filterwarnings("ignore", message="enable_nested_tensor")  # 隐藏不影响本实验的嵌套张量提示
torch.manual_seed(42)  # 固定模型初始化与训练批次顺序
np.random.seed(42)  # 固定绘图中可能使用的 NumPy 随机过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 构造只有顺序不同的历史

每段历史包含 10 个时间步。第一个通道在“接触”时为 1，第二个通道在“抬起”时为 1，第三个通道加入很小的传感器噪声。标签 1 表示先接触后抬起，标签 0 表示先抬起后接触。

In [ ]:
sequence_length = 10  # 设置每段机器人历史包含十个时间步
def make_histories(sample_count, random_seed):  # 定义生成事件顺序数据的函数
    generator = torch.Generator().manual_seed(random_seed)  # 为当前数据集建立独立随机生成器
    histories = torch.zeros(sample_count, sequence_length, 3)  # 预先分配接触、抬起和噪声三个通道
    labels = torch.zeros(sample_count, dtype=torch.long)  # 预先分配二分类标签
    for sample_index in range(sample_count):  # 逐条构造具有不同事件位置的历史
        contact_step, lift_step = torch.randperm(sequence_length, generator=generator)[:2].tolist()  # 随机选择两个不同时间位置
        histories[sample_index, contact_step, 0] = 1.0  # 在第一个通道记录接触事件
        histories[sample_index, lift_step, 1] = 1.0  # 在第二个通道记录抬起事件
        histories[sample_index, :, 2] = 0.03 * torch.randn(sequence_length, generator=generator)  # 加入很小的传感器噪声
        labels[sample_index] = int(contact_step < lift_step)  # 根据真实先后顺序产生监督标签
    return histories, labels  # 返回历史张量与顺序标签
train_histories, train_labels = make_histories(1200, 11)  # 生成用于参数更新的训练集
test_histories, test_labels = make_histories(400, 12)  # 使用不同随机种子生成未见测试集
example_indices = [int(torch.where(test_labels == 1)[0][0]), int(torch.where(test_labels == 0)[0][0])]  # 各选择一个正序和逆序样本
fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True)  # 创建两段事件历史的可视化
for axis, example_index in zip(axes, example_indices):  # 依次绘制正序样本和逆序样本
    contact_values = test_histories[example_index, :, 0].numpy()  # 取出当前样本的接触事件通道
    lift_values = test_histories[example_index, :, 1].numpy()  # 取出当前样本的抬起事件通道
    axis.stem(range(sequence_length), contact_values, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="Contact")  # 用蓝色脉冲显示接触时刻
    axis.stem(range(sequence_length), lift_values, linefmt="C1-", markerfmt="C1s", basefmt=" ", label="Lift")  # 用橙色脉冲显示抬起时刻
    axis.set(ylabel="Event", yticks=[0, 1], title=f"Label = {int(test_labels[example_index])}")  # 标明事件值和真实标签
    axis.legend(loc="upper right")  # 显示两类事件的图例
axes[-1].set_xlabel("Time step")  # 为共用横轴标记时间步
fig.suptitle("The same events can describe different temporal processes")  # 强调事件集合相同但顺序不同
fig.tight_layout()  # 调整子图间距避免标题重叠
plt.show()  # 显示两种顺序的机器人历史

**怎样理解结果：** 两段历史都只包含一次 Contact 和一次 Lift，事件数量与事件类型完全相同。第一段中接触先发生，因此标签为 1；第二段中抬起先发生，因此标签为 0。模型若只知道“出现了哪些事件”，却不知道它们位于哪个时间步，就无法可靠地区分这两个过程。

## 2. 只改变位置编码，比较两个 Transformer

两个模型使用相同的输入投影、单层 Transformer Encoder、平均池化和分类头。唯一差别是第二个模型会给每个时间步加入可学习位置向量。平均池化使无位置模型只能利用事件集合，不能从最终输出恢复排列顺序。

In [ ]:
class OrderTransformer(nn.Module):  # 定义用于判断事件顺序的小型 Transformer
    def __init__(self, use_position):  # 根据开关决定是否启用位置编码
        super().__init__()  # 初始化 PyTorch 模型基类
        self.input_projection = nn.Linear(3, 24)  # 把三个传感器通道投影到二十四维 token
        self.position = nn.Parameter(torch.zeros(1, sequence_length, 24)) if use_position else None  # 为每个时间步建立可学习位置向量
        encoder_layer = nn.TransformerEncoderLayer(d_model=24, nhead=4, dim_feedforward=48, dropout=0.0, batch_first=True, norm_first=True)  # 建立四头 Pre-Norm Transformer Block
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)  # 使用一层编码器更新整段历史
        self.classifier = nn.Linear(24, 2)  # 把汇总后的上下文映射到两个顺序类别
    def forward(self, histories):  # 定义从历史输入到分类分数的前向计算
        tokens = self.input_projection(histories)  # 把每个时间步转换成统一维度的 token
        if self.position is not None:  # 检查当前模型是否允许读取时间位置
            tokens = tokens + self.position  # 把可学习位置向量加入各时间步 token
        encoded_tokens = self.encoder(tokens)  # 使用 Self-Attention 交换整段历史的信息
        context = encoded_tokens.mean(dim=1)  # 对所有时间位置平均得到全局上下文
        return self.classifier(context)  # 输出两个类别的未归一化分数
def train_model(use_position, epoch_count=35):  # 定义训练并记录测试表现的函数
    torch.manual_seed(42)  # 让两个实验从可比较的参数初始化开始
    model = OrderTransformer(use_position)  # 创建有位置或无位置的模型
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)  # 使用 Adam 更新全部可学习参数
    loss_function = nn.CrossEntropyLoss()  # 使用交叉熵监督两个顺序类别
    loss_history = []  # 保存每轮训练结束后的平均损失
    accuracy_history = []  # 保存每轮在未见测试集上的准确率
    for epoch_index in range(epoch_count):  # 重复多轮遍历训练数据
        shuffled_indices = torch.randperm(len(train_histories))  # 每轮随机打乱样本顺序
        batch_losses = []  # 暂存当前轮各批次损失
        for start_index in range(0, len(train_histories), 120):  # 每次使用一百二十段历史更新模型
            batch_indices = shuffled_indices[start_index:start_index + 120]  # 取出当前批次的样本索引
            predictions = model(train_histories[batch_indices])  # 对当前批次历史执行前向预测
            loss = loss_function(predictions, train_labels[batch_indices])  # 计算预测顺序与真实标签的交叉熵
            optimizer.zero_grad()  # 清除上一个批次残留的参数梯度
            loss.backward()  # 反向传播计算各参数应调整的方向
            optimizer.step()  # 按学习率更新 Transformer 参数
            batch_losses.append(float(loss.detach()))  # 保存当前批次损失供轮次汇总
        model.eval()  # 切换到评估模式计算未见数据表现
        with torch.no_grad():  # 关闭评估过程中的梯度记录
            test_predictions = model(test_histories).argmax(dim=1)  # 得到测试集上的离散顺序判断
            test_accuracy = (test_predictions == test_labels).float().mean().item()  # 计算测试准确率
        model.train()  # 恢复训练模式以便继续下一轮更新
        loss_history.append(float(np.mean(batch_losses)))  # 记录当前轮平均训练损失
        accuracy_history.append(test_accuracy)  # 记录当前轮测试准确率
    return model, loss_history, accuracy_history  # 返回训练模型和两条学习曲线
model_without_position, loss_without_position, accuracy_without_position = train_model(False)  # 训练不能读取时间位置的基线模型
model_with_position, loss_with_position, accuracy_with_position = train_model(True)  # 训练加入可学习位置编码的模型

In [ ]:
epochs = np.arange(1, len(loss_with_position) + 1)  # 建立从一开始的训练轮次横轴
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))  # 创建损失与准确率两个坐标轴
axes[0].plot(epochs, loss_without_position, label="Without position", color="#94a3b8")  # 绘制无位置模型的训练损失
axes[0].plot(epochs, loss_with_position, label="With position", color="#2563eb")  # 绘制有位置模型的训练损失
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")  # 标注损失曲线的含义
axes[1].plot(epochs, accuracy_without_position, label="Without position", color="#94a3b8")  # 绘制无位置模型的测试准确率
axes[1].plot(epochs, accuracy_with_position, label="With position", color="#2563eb")  # 绘制有位置模型的测试准确率
axes[1].axhline(0.5, color="#dc2626", linestyle="--", label="Random guess")  # 标出平衡二分类的随机猜测水平
axes[1].set(title="Accuracy on unseen histories", xlabel="Epoch", ylabel="Accuracy", ylim=(0.4, 1.05))  # 标注未见数据准确率范围
for axis in axes:  # 统一处理两个子图的图例和网格
    axis.legend()  # 显示有无位置编码的曲线名称
    axis.grid(alpha=0.2)  # 添加淡网格方便读取曲线数值
fig.tight_layout()  # 调整子图间距避免文字重叠
plt.show()  # 显示两个模型的完整学习过程

**怎样理解结果：** 无位置模型的损失停留在约 $\log 2$，测试准确率也在 50% 附近，因为它看到的始终只是“一次接触加一次抬起”这个无序集合。有位置模型则能把事件特征与时间位置联系起来，损失迅速下降，并能在未见事件位置上正确判断先后关系。这里并不是位置编码单独完成分类，而是位置编码为 Attention 提供了建立顺序关系所需的信息。

## 3. 检查两类错误分别发生在哪里

准确率只能给出一个总数。下面计算两个模型的混淆矩阵，观察它们是否同时学会了“先接触”和“先抬起”两种情况。

In [ ]:
def confusion_matrix_for(model):  # 定义计算二分类混淆矩阵的函数
    model.eval()  # 切换到评估模式保证结果稳定
    with torch.no_grad():  # 关闭推理过程中的梯度记录
        predicted_labels = model(test_histories).argmax(dim=1)  # 得到全部测试历史的预测标签
    matrix = np.zeros((2, 2), dtype=int)  # 建立真实类别乘预测类别的计数矩阵
    for true_label, predicted_label in zip(test_labels.numpy(), predicted_labels.numpy()):  # 逐条累计预测结果
        matrix[true_label, predicted_label] += 1  # 把当前样本计入对应真实与预测单元格
    return matrix  # 返回二乘二混淆矩阵
matrices = [confusion_matrix_for(model_without_position), confusion_matrix_for(model_with_position)]  # 分别统计两个模型的类别判断
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.7))  # 创建两个混淆矩阵坐标轴
for axis, matrix, title in zip(axes, matrices, ["Without position", "With position"]):  # 依次绘制无位置和有位置结果
    image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=matrix.max())  # 使用相同蓝色系显示样本计数
    axis.set(title=title, xlabel="Predicted label", ylabel="True label")  # 标注矩阵行列的类别含义
    axis.set_xticks([0, 1])  # 显示两个预测类别刻度
    axis.set_yticks([0, 1])  # 显示两个真实类别刻度
    for row in range(2):  # 遍历两个真实类别
        for column in range(2):  # 遍历两个预测类别
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center", color="black")  # 写出当前单元格的样本数量
fig.colorbar(image, ax=axes, shrink=0.82, label="Sample count")  # 添加表示样本数量的共用颜色条
plt.show()  # 显示两个模型的分类结构

**怎样理解结果：** 无位置模型无法稳定区分两个类别，预测会集中或近似随机分配到某一类；有位置模型的样本则集中在主对角线，说明两种事件顺序都能识别，而不是只记住类别比例。

**本练习的结论：** Self-Attention 擅长比较和汇集 token，却不会凭空知道 token 的先后位置。机器人历史模型必须显式提供位置、时间戳或时间间隔，并根据预测方式正确设置 causal mask。否则模型即使看到了所有事件，也可能无法理解动作过程。